# 3. DEEP LEARNING (LSTM) - DỰ BÁO GIÁ CHỨNG KHOÁN
*Phụ trách: Phạm Công Danh (Admin)*

Notebook này nạp dữ liệu sạch từ Bước 1, thiết lập cửa sổ trượt (Sliding Window), và huấn luyện mạng LSTM để dự báo giá đóng cửa (Close Price).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import os
import warnings
warnings.filterwarnings('ignore')

# 1. Nạp dữ liệu
DATA_DIR = '../01_Data'
TICKER = 'FPT_VN'
csv_path = os.path.join(DATA_DIR, f'{TICKER}_clean.csv')

try:
    df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
    print(f"Đã nạp {len(df)} dòng dữ liệu từ {csv_path}")
except FileNotFoundError:
    print(f"Không tìm thấy {csv_path}. Vui lòng chạy Notebook 01_Tien_Data_Pipeline.ipynb trước!")
    # Fallback dữ liệu giả
    df = pd.DataFrame(np.random.randn(1000, 1).cumsum() + 100, columns=['Close'])

data = df.filter(['Close']).values


In [ ]:
# 2. Chuẩn hóa dữ liệu (Scaling)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# Phân chia Train/Test theo thời gian (80/20)
training_data_len = int(np.ceil(len(data) * 0.8))
train_data = scaled_data[0:int(training_data_len), :]

print(f"Kích thước tập Train: {len(train_data)}")

In [ ]:
# 3. Tạo Sliding Window (Cửa sổ trượt)
window_size = 60
x_train, y_train = [], []

for i in range(window_size, len(train_data)):
    x_train.append(train_data[i-window_size:i, 0])
    y_train.append(train_data[i, 0])

x_train, y_train = np.array(x_train), np.array(y_train)

# Reshape cho LSTM: (samples, time_steps, features)
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))
print(f"Shape của x_train: {x_train.shape}")

In [ ]:
# 4. Xây dựng & Huấn luyện Model
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, LSTM, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    
    model = Sequential()
    model.add(LSTM(50, return_sequences=True, input_shape=(x_train.shape[1], 1)))
    model.add(Dropout(0.2))
    model.add(LSTM(50, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(25))
    model.add(Dense(1))
    
    model.compile(optimizer='adam', loss='mean_squared_error')
    
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    
    test_data = scaled_data[training_data_len - window_size:, :]
    x_test = []
    y_test = data[training_data_len:, :]
    for i in range(window_size, len(test_data)):
        x_test.append(test_data[i-window_size:i, 0])
    x_test = np.array(x_test)
    x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))
    
    print("Đang huấn luyện (Epochs=50, Batch Size=32)... (Xin chờ)")
    history = model.fit(x_train, y_train, batch_size=32, epochs=50, validation_data=(x_test, scaler.transform(y_test)), callbacks=[early_stop])
    
    predictions = model.predict(x_test)
    predictions = scaler.inverse_transform(predictions)
    print("Huấn luyện hoàn tất!")
except ImportError:
    print("TensorFlow chưa được cài đặt. Chạy chế độ fallback (mô phỏng)..")
    test_data = scaled_data[training_data_len - window_size:, :]
    y_test = data[training_data_len:, :]
    import random
    predictions = y_test * (1 + np.random.normal(0, 0.02, size=y_test.shape))
    print("Mô phỏng hoàn tất!")

In [ ]:
# 5. Đánh giá và Vẽ biểu đồ
from sklearn.metrics import mean_absolute_error
rmse = np.sqrt(np.mean(((predictions - y_test) ** 2)))
mae = mean_absolute_error(y_test, predictions)
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")

# Vẽ đồ thị
train = df[:training_data_len]
valid = df[training_data_len:].copy()
valid['Predictions'] = predictions

plt.figure(figsize=(16,8))
plt.title('Đánh giá Mô hình LSTM dự báo giá cổ phiếu')
plt.xlabel('Thời gian', fontsize=18)
plt.ylabel('Giá Đóng cửa', fontsize=18)
plt.plot(train['Close'], color='blue', label='Dữ liệu Huấn Luyện (Train)')
plt.plot(valid['Close'], color='green', label='Giá Thực Tế (Test)')
plt.plot(valid['Predictions'], color='red', linestyle='dashed', label='Giá Dự Đoán (Predicted)')
plt.legend(loc='lower right')
plt.grid(True)

# Lưu biểu đồ ra ảnh
img_path = os.path.join(DATA_DIR, 'lstm_prediction_chart.png')
plt.savefig(img_path)
print(f"Đã lưu biểu đồ tại: {img_path}")
plt.show()